# Task A (Hate / Non-Hate): smoke test (run this first, about 30 minutes)

Runs every stage of the reproduction on a small sample before you spend 20 to 30 GPU-hours on the full sessions: data checksums and holdout fingerprint, loading Gemma-4-12B in 4 bits, a TAPT pass, two R-Drop classifiers started from the TAPT adapter (one per GPU), reloading a saved classifier and checking it reproduces its predictions, combining the two, and packaging a predictions file. It ends with a PASS/FAIL checklist. The models train for one epoch on about 100 comments, so their predictions are not meant to be accurate and the ZIP it writes is not a submission.

**Before you run** (Kaggle notebook settings):

1. *Accelerator*: **GPU T4 x2**. *Internet*: **on**.
2. *Add Input*: your private Kaggle dataset holding the six organizer CSVs and
   `offenseval_kn.csv` (see `reproduce/README.md`; the files are found by name and checked
   against their recorded SHA-256, wherever they sit in the dataset).
3. Optional: *Add-ons -> Secrets* -> `HF_TOKEN` (a Hugging Face token raises download
   limits; the model itself is openly available).
4. Use **Save Version -> Save & Run All (Commit)**. An interactive session stops when the
   browser tab closes.


In [ ]:
TASK = "a"


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Aaryan127/ProjectMANAS-HastikaICON2026.git"
REPO_REF = "v0.3.1"          # the release this notebook was written for
REPO = Path("/kaggle/working/ProjectMANAS-HastikaICON2026")
WORK = Path("/kaggle/working")

def run(*cmd):
    """Run a command and stream its output into this cell; stop on failure."""
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}")

if not REPO.exists():
    run("git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO)
os.chdir(REPO)
run(sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[llm]")
sys.path.insert(0, str(REPO / "src"))

In [ ]:
# google/gemma-4-12B is openly available (Apache 2.0); no token is needed. If a Kaggle secret
# called HF_TOKEN exists (Add-ons -> Secrets), it is used for higher download limits.
# Never paste a token into a cell.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception as error:
    print("No HF_TOKEN secret found:", error)
run(sys.executable, "-m", "hastika.reproduce", "check")

In [ ]:
run(sys.executable, "-m", "hastika.reproduce", "smoke", "--task", TASK,
    "--work", WORK, "--search", "/kaggle/input")